In [ ]:
%%writefile h2o.c
/*
SO - 2025/02
Henrique Luz Alves Coutinho
791265

Kailayni Rodrigues Janez
824751

Eduardo da Silva Ribeiro
833021
*/
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <signal.h>
#include <unistd.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <time.h>
#include <errno.h>
// Bibliotecas POSIX
#include <pthread.h>
#include <semaphore.h>

// Sincronização
pthread_mutex_t mutex;
sem_t H_sem;
sem_t O_sem;
pthread_barrier_t barrier;

// Contadores Globais (compartilhado pelos threads)
int H_count = 0;
int O_count = 0;
int H_id = 0;
int O_id = 0;

// Variáveis do Clock
struct timespec ultimo_update, esse_update;
float tempo_decorrido;

// Handler
void trata_SIGINT(int signum) {
  printf("\nTerminando o gerador...\n");

  pthread_mutex_destroy(&mutex);
  sem_destroy(&H_sem);
  sem_destroy(&O_sem);
  pthread_barrier_destroy(&barrier);

  exit(0);
}

void bond() {
    printf("Molécula H2O formada!\n");
    fflush(stdout);
}

// Thread Hidrogênio
void* hydrogen(void* arg) {
    int id = *(int*)arg;

    // Entra na região crítica
    pthread_mutex_lock(&mutex);
    H_count++;
    printf("H%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    // Verifica se pode formar molécula
    if (H_count >= 2 && O_count >= 1) {
        // Pode formar molécula!
        sem_post(&H_sem);  // Acorda outro H
        sem_post(&O_sem);  // Acorda um O

        // Atualiza contadores
        H_count -= 2;
        O_count -= 1;
        printf("H%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);

        pthread_mutex_unlock(&mutex);
    } else {
        // Não pode formar ainda, precisa esperar
        pthread_mutex_unlock(&mutex);

        // Espera ser sinalizado
        sem_wait(&H_sem);
    }

    pthread_barrier_wait(&barrier);
    // bond();
    free(arg);
    return NULL;
}

// Thread Oxigênio
void* oxygen(void* arg) {
    int id = *(int*)arg;

    // Entra na região crítica
    pthread_mutex_lock(&mutex);
    O_count++;
    printf("O%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    // Verifica se pode formar molécula
    if (H_count >= 2 && O_count >= 1) {
        // Pode formar molécula!
        sem_post(&H_sem);  // Acorda primeiro H
        sem_post(&H_sem);  // Acorda segundo H

        // Atualiza contadores
        H_count -= 2;
        O_count -= 1;
        printf("O%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);

        pthread_mutex_unlock(&mutex);
    } else {
        // Não pode formar ainda, precisa esperar
        pthread_mutex_unlock(&mutex);

        // Espera ser sinalizado
        sem_wait(&O_sem);
    }

    pthread_barrier_wait(&barrier);
    bond();
    free(arg);
    return NULL;
}

int main(int argc, char* argv[]) {
  int ph, tick_ms;
  pthread_t thread;
  signal(SIGINT, trata_SIGINT);

  if (argv[1] != NULL && argv[2] != NULL) {
    // Com Argumentos
    ph = atoi(argv[1]);
    tick_ms = atoi(argv[2]);
  } else {
    // Sem Argumentos
    printf("Inicializado sem argumentos ou houve falha na leitura.\n");
    printf("Argumentos são \\.h20 [ph] [tick]\n");
    printf("ph determina a diferença na geração de H vs O. Padrão é 2, ou seja, 2 H para cada O.\n");
    printf("tick determina a velocidade em ms de update. Padrão é 1000, ou seja, é gerado um H ou O a cada 1s.\n");
    printf("tick = 0 faz com que se forme threads à todo momento.\n");
    // Valores default
    ph = 2;
    tick_ms = 1000;
  }
  printf("Configuração: ph=%d, tick=%dms\n\n", ph, tick_ms);

  // Inicializa o randomizador
  srand(time(NULL));

  // Inicializa o Clock
  if (clock_gettime(CLOCK_REALTIME, &ultimo_update) < 0) {
    printf("Erro em clock_gettime");
    return -1;
  }

  // Inicializa as primitivas de sincronização
  pthread_mutex_init(&mutex, NULL);
  sem_init(&H_sem, 0, 0);
  sem_init(&O_sem, 0, 0);
  pthread_barrier_init(&barrier, NULL, 3);

  while (1) {
    // Loop Principal

    if (clock_gettime(CLOCK_REALTIME, &esse_update) < 0) {
      printf("Erro em clock_gettime");
      return -1;
    }

    // Checar Update e sair se não atingiu o parametro tick
    long tempo_decorrido_ms = (esse_update.tv_sec - ultimo_update.tv_sec) * 1000 +
                              (esse_update.tv_nsec - ultimo_update.tv_nsec) / 1000000;
    if (tick_ms != 0 && tempo_decorrido_ms < tick_ms) {
      usleep(1000);
      continue;
    }

    // A partir desse ponto, começa a geração de um átomo
    ultimo_update = esse_update;

    // Gera um número aleatório para decidir H ou O
    int r = rand() % 100;

    // Probabilidade baseada em ph
    // Se ph=2, queremos 2 H para cada 1 O, ou seja, 66% H, 33% O
    int prob_h = (ph * 100) / (ph + 1);

    if (r < prob_h) {
      // Cria thread Hidrogênio
      int* id = malloc(sizeof(int));
      *id = H_id++;
      if (pthread_create(&thread, NULL, hydrogen, id) != 0) {
        perror("Erro ao criar thread hydrogen");
        free(id);
        } else {
          pthread_detach(thread); // Detach para não precisar fazer join
        }
    } else {
      // Cria thread Oxigênio
      int* id = malloc(sizeof(int));
      *id = O_id++;
      if (pthread_create(&thread, NULL, oxygen, id) != 0) {
        perror("Erro ao criar thread oxygen");
        free(id);
      } else {
        pthread_detach(thread); // Detach para não precisar fazer join
      }
    }
  }

  return 0;
}


Writing h2o.c


In [ ]:
!gcc -o h2o h2o.c -lpthread

In [ ]:
!./h2o 10 50

Configuração: ph=10, tick=50ms

H0 chega. Esperando: H=1, O=0
H1 chega. Esperando: H=2, O=0
H2 chega. Esperando: H=3, O=0
H3 chega. Esperando: H=4, O=0
H4 chega. Esperando: H=5, O=0
H5 chega. Esperando: H=6, O=0
H6 chega. Esperando: H=7, O=0
H7 chega. Esperando: H=8, O=0
H8 chega. Esperando: H=9, O=0
O0 chega. Esperando: H=9, O=1
O0: Formando molécula! Novos valores: H=7, O=0
Molécula H2O formada!
H9 chega. Esperando: H=8, O=0
H10 chega. Esperando: H=9, O=0
O1 chega. Esperando: H=9, O=1
O1: Formando molécula! Novos valores: H=7, O=0
Molécula H2O formada!
H11 chega. Esperando: H=8, O=0
H12 chega. Esperando: H=9, O=0
H13 chega. Esperando: H=10, O=0
H14 chega. Esperando: H=11, O=0
H15 chega. Esperando: H=12, O=0
H16 chega. Esperando: H=13, O=0
O2 chega. Esperando: H=13, O=1
O2: Formando molécula! Novos valores: H=11, O=0
Molécula H2O formada!
O3 chega. Esperando: H=11, O=1
O3: Formando molécula! Novos valores: H=9, O=0
Molécula H2O formada!
O4 chega. Esperando: H=9, O=1
O4: Formando moléc

Começamos inicializando as variáveis globais (compartilhadas entres as threads), primeiro as de sincronização, seguidas por contadores de ambos tipos de threads, ids temporários e variáveis de type.h que formam a estrutura que mede o tempo para formar threads novas.
```
pthread_mutex_t mutex;
sem_t H_sem;
sem_t O_sem;
pthread_barrier_t barrier;

int H_count = 0;
int O_count = 0;
int H_id = 0;
int O_id = 0;

struct timespec ultimo_update, esse_update;
float tempo_decorrido;
```
Aqui criamos um handler de SIGINT (Ctrl+C) para o programa resolver os semáforos do kernel antes de fechar. O que é muito importante, pois esses são alocados em memória do kernel, compartilhada entre todos os processos.
```
void trata_SIGINT(int signum) {
  printf("\nTerminando o gerador...\n");

  pthread_mutex_destroy(&mutex);
  sem_destroy(&H_sem);
  sem_destroy(&O_sem);
  pthread_barrier_destroy(&barrier);

  exit(0);
}
```
A função bond() apenas serve para transmitir ao usuário que uma ligação foi feita. Em uma implementação que não seja apenas educacional, essa função provavelmente receberia as threads involvidas como parametros. No ponto que bond() é chamado, as três threads que fazem parte da "molécula" estão fora das filas do semáforo e, portanto, fora da região crítica e protegidas de problemas de sincronização.
```
void bond() {
    printf("Molécula H2O formada!\n");
    fflush(stdout);
}
```
As duas funções das threads são bem parecidas. Ao serem formadas, estes entram na região críticas, acionando o mutex, e checam se é possível formar uma molécula com os átomos atualmente em espera. Se é possível, com 2 Hidrogênios e 1 Oxigênio em espera, o novo átomo acorda os outros threads com sem_post() e forma a molécula. Se não é possível, este mesmo átomo que chegou desliga o mutex e entra em espera no semáforo.

Há algumas observações a serem feitas aqui:
Primeiro, ao chamarem sem_wait(), a execução entra em bloqueio precisamente na linha que a chamada foi realizada. Assim, quando acordamos uma thread, só há uma coisa que pode ter acontecido, que é a formação de uma molécula. Assim, essas já "sabem" o que devem fazer, e não precisam checar novamente os contadores.
Segundo, a uma condição de corrida que requer o uso de dois níveis de sincronização. O mutex é usado para os contadores apenas, enquanto a barreira é usada para sincronizar a existência dos 3 threads necessários para a formação do H2O. Note que temos que desengagar o mutex ANTES de sem_wait(), porque neste ponto a execução é bloqueada. Mas se desligamos o mutex antes de estar na fila, há a possibilidade de que um outro thread seja criado neste meio-termo e detecte erroneamente que temos o número necessário de threads nas filas.
```
void* hydrogen(void* arg) {
    int id = *(int*)arg;

    // Entra na região crítica
    pthread_mutex_lock(&mutex);
    H_count++;
    printf("H%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    // Verifica se pode formar molécula
    if (H_count >= 2 && O_count >= 1) {
        // Pode formar molécula!
        sem_post(&H_sem);  // Acorda outro H
        sem_post(&O_sem);  // Acorda um O

        // Atualiza contadores
        H_count -= 2;
        O_count -= 1;

        pthread_mutex_unlock(&mutex);
    } else {
        // Não pode formar ainda, precisa esperar
        pthread_mutex_unlock(&mutex);

        // Espera ser sinalizado
        sem_wait(&H_sem);
    }

    pthread_barrier_wait(&barrier);
    free(arg);
    return NULL;
}

// Thread Oxigênio
void* oxygen(void* arg) {
    int id = *(int*)arg;

    // Entra na região crítica
    pthread_mutex_lock(&mutex);
    O_count++;
    printf("O%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    // Verifica se pode formar molécula
    if (H_count >= 2 && O_count >= 1) {
        // Pode formar molécula!
        sem_post(&H_sem);  // Acorda primeiro H
        sem_post(&H_sem);  // Acorda segundo H

        // Atualiza contadores
        H_count -= 2;
        O_count -= 1;
        
        pthread_mutex_unlock(&mutex);
    } else {
        // Não pode formar ainda, precisa esperar
        pthread_mutex_unlock(&mutex);

        // Espera ser sinalizado
        sem_wait(&O_sem);
    }

    pthread_barrier_wait(&barrier);
    printf("O%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);
    bond();
    free(arg);
    return NULL;
}
```
De fato, aqui se encontra a condição de corrida. Novamente, se há uma preempção neste ponto crítico, há a possibilidade do seguinte ocorrer:

1. Há H = 1 e O = 0 no início
2. Entra um H, muda os contadores, mas é interrompido antes de entrar na fila do semáforo
3. Entra um O, e, vendo que nos contadores há dois H, aquele tenta acordar dois Hs, mas acaba só acordando um e formando, ao invés de um H2O, um HO-
4. H anterior continua a execução, agora entrando na fila, mas os contadores já mudaram, então este poderá nunca ser detectado e talvez esperará para sempre.
```
// Contadores mudam aqui em cima
} else {
    // Não pode formar ainda, precisa esperar
    pthread_mutex_unlock(&mutex);

    // Ponto de condição de corrida

    // Espera ser sinalizado
    sem_wait(&O_sem);
}
```
Este problema precisa da criação de um outro nível de sincronização, usando a primitiva pthread_barrier_t. Antes de se comprometer à formar um H2O, cada thread espera na barreira, e quando 3 desses esperam na barreira, somente então uma molécula é formada.

Assim, fazemos questão que realmente tenha 3 threads formando uma molécula, e evitando casos como OH-
```
// Cada thread passa por uma barreira
pthread_barrier_wait(&barrier);

// No main, a barreira é inicializada
pthread_barrier_init(&barrier, NULL, 3);
```
Na main, temos 2 argumentos, ph e tick.

ph determina a diferença na produção de Hidrogênio vs Oxigênio. Um valor de 2 significa 2 Hs para cada O formado.
tick é parte de um sistema de update em ms. No loop infinito principal, o processo compara o tempo atual com o timestamp da formação da última thread. Assim, a cada tempo = tick que passou, há a formação de outra thread. Com tick = 1000, há a formação de uma nova thread a cada 1s. Com tick = 0, o processo não usa o sistema de update, e cria threads infinitamente ou até ser bloqueado por falta de recursos.
```
int main(int argc, char* argv[]) {
  int ph, tick_ms;
  pthread_t thread;
  signal(SIGINT, trata_SIGINT);

  if (argv[1] != NULL && argv[2] != NULL) {
    // Com Argumentos
    ph = atoi(argv[1]);
    tick_ms = atoi(argv[2]);
  } else {
    // Sem Argumentos
    printf("Inicializado sem argumentos ou houve falha na leitura.\n");
    printf("Argumentos são \\.h20 [ph] [tick]\n");
    printf("ph determina a diferença na geração de H vs O. Padrão é 2, ou seja, 2 H para cada O.\n");
    printf("tick determina a velocidade em ms de update. Padrão é 1000, ou seja, é gerado um H ou O a cada 1s.\n");
    printf("tick = 0 faz com que se forme threads à todo momento.\n");
    // Valores default
    ph = 2;
    tick_ms = 1000;
  }
  printf("Configuração: ph=%d, tick=%dms\n\n", ph, tick_ms);

  // Inicializa o randomizador
  srand(time(NULL));

  // Inicializa o Clock
  if (clock_gettime(CLOCK_REALTIME, &ultimo_update) < 0) {
    printf("Erro em clock_gettime");
    return -1;
  }

  // Inicializa as primitivas de sincronização
  pthread_mutex_init(&mutex, NULL);
  sem_init(&H_sem, 0, 0);
  sem_init(&O_sem, 0, 0);
  pthread_barrier_init(&barrier, NULL, 3);

  while (1) {
    // Loop Principal

    if (clock_gettime(CLOCK_REALTIME, &esse_update) < 0) {
      printf("Erro em clock_gettime");
      return -1;
    }

    // Checar Update e sair se não atingiu o parametro tick
    long tempo_decorrido_ms = (esse_update.tv_sec - ultimo_update.tv_sec) * 1000 +
                              (esse_update.tv_nsec - ultimo_update.tv_nsec) / 1000000;
    if (tick_ms != 0 && tempo_decorrido_ms < tick_ms) {
      usleep(1000); // Dorme por 1ms
      continue;
    }

    // A partir desse ponto, começa a geração de um átomo
    ultimo_update = esse_update;

    // Gera um número aleatório para decidir H ou O
    int r = rand() % 100;

    // Probabilidade baseada em ph
    // Se ph=2, queremos 2 H para cada 1 O, ou seja, 66% H, 33% O
    int prob_h = (ph * 100) / (ph + 1);

    if (r < prob_h) {
      // Cria thread Hidrogênio
      int* id = malloc(sizeof(int));
      *id = H_id++;
      if (pthread_create(&thread, NULL, hydrogen, id) != 0) {
        perror("Erro ao criar thread hydrogen");
        free(id);
        } else {
          pthread_detach(thread); // Detach para não precisar fazer join
        }
    } else {
      // Cria thread Oxigênio
      int* id = malloc(sizeof(int));
      *id = O_id++;
      if (pthread_create(&thread, NULL, oxygen, id) != 0) {
        perror("Erro ao criar thread oxygen");
        free(id);
      } else {
        pthread_detach(thread); // Detach para não precisar fazer join
      }
    }
  }

  return 0;
}
```

Apresentação: https://docs.google.com/presentation/d/1tPoYEUW3MFopXHi4bD_iR7_LZdb_BPScomjl3KYgDos/edit?usp=sharing

# Segunda Solução

In [ ]:
%%writefile h2o_final.c
/*
SO - 2025/02
Henrique Luz Alves Coutinho
791265

Kailayni Rodrigues Janez
824751

Eduardo da Silva Ribeiro
833021
*/
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <signal.h>
#include <unistd.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <time.h>
#include <errno.h>
// Bibliotecas POSIX
#include <pthread.h>
#include <semaphore.h>

// Sincronização
pthread_mutex_t mutex;
pthread_mutex_t barrier_mutex;
// Sem Semáforos
// sem_t H_sem;
// sem_t O_sem;
pthread_cond_t H_cond;
pthread_cond_t O_cond;
pthread_barrier_t barrier;

// Contadores Globais (compartilhado pelos threads)
int H_count = 0;
int O_count = 0;
int H_id = 0;
int O_id = 0;
int barrier_H_count = 0;
int barrier_O_count = 0;

// Variáveis do Clock
struct timespec ultimo_update, esse_update;
float tempo_decorrido;

// Handler
void trata_SIGINT(int signum) {
  printf("\nTerminando o gerador...\n");

  pthread_mutex_destroy(&barrier_mutex);
  pthread_mutex_destroy(&mutex);
  pthread_cond_destroy(&H_cond);
  pthread_cond_destroy(&O_cond);
  pthread_barrier_destroy(&barrier);

  exit(0);
}

void bond() {
    printf("Molécula H2O formada!\n");
    fflush(stdout);
}

#define HYDROGEN 1
#define OXYGEN 0

void barreira(int type) {
    pthread_mutex_lock(&barrier_mutex);
    if (type == OXYGEN)
        barrier_O_count++;
    else
        barrier_H_count++;
    pthread_mutex_unlock(&barrier_mutex);

    int result = pthread_barrier_wait(&barrier);

    if (result == PTHREAD_BARRIER_SERIAL_THREAD) {
        pthread_mutex_lock(&barrier_mutex);

        if (barrier_H_count != 2 || barrier_O_count != 1) {
            printf("ERRO FATAL! Composição incorreta: H=%d, O=%d\n",
                   barrier_H_count, barrier_O_count);
            pthread_mutex_unlock(&barrier_mutex);
            exit(1);
        }

        barrier_H_count = 0;
        barrier_O_count = 0;

        pthread_mutex_unlock(&barrier_mutex);

        bond();
    }
}

// Thread Hidrogênio
void* hydrogen(void* arg) {
    int id = *(int*)arg;

    pthread_mutex_lock(&mutex);
    H_count++;
    printf("H%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    if (H_count >= 2 && O_count >= 1) {
        // Inicia a formação dá molécula
        H_count -= 2;
        O_count -= 1;
        printf("H%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);

        // Acorda as outras threads
        pthread_cond_signal(&H_cond);
        pthread_cond_signal(&O_cond);

        pthread_mutex_unlock(&mutex);
    } else {
        // Não há threads o suficiente, espera
        pthread_cond_wait(&H_cond, &mutex);
        // Ao acordar, desliga o mutex se não trava ao tentar formar molécula
        pthread_mutex_unlock(&mutex);
    }

    barreira(HYDROGEN);

    free(arg);
    return NULL;
}

// Thread Oxigênio
void* oxygen(void* arg) {
    int id = *(int*)arg;

    pthread_mutex_lock(&mutex);
    O_count++;
    printf("O%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    if (H_count >= 2 && O_count >= 1) {
        H_count -= 2;
        O_count -= 1;
        printf("H%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);

        pthread_cond_signal(&H_cond);
        pthread_cond_signal(&H_cond);

        pthread_mutex_unlock(&mutex);
    } else {
        pthread_cond_wait(&O_cond, &mutex);
        pthread_mutex_unlock(&mutex);
    }

    barreira(OXYGEN);

    free(arg);
    return NULL;
}

int main(int argc, char* argv[]) {
  int ph, tick_ms;
  pthread_t thread;
  signal(SIGINT, trata_SIGINT);

  if (argv[1] != NULL && argv[2] != NULL) {
    // Com Argumentos
    ph = atoi(argv[1]);
    tick_ms = atoi(argv[2]);
  } else {
    // Sem Argumentos
    printf("Inicializado sem argumentos ou houve falha na leitura.\n");
    printf("Argumentos são \\.h20 [ph] [tick]\n");
    printf("ph determina a diferença na geração de H vs O. Padrão é 2, ou seja, 2 H para cada O.\n");
    printf("tick determina a velocidade em ms de update. Padrão é 1000, ou seja, é gerado um H ou O a cada 1s.\n");
    printf("tick = 0 faz com que se forme threads à todo momento.\n");
    // Valores default
    ph = 2;
    tick_ms = 1000;
  }
  printf("Configuração: ph=%d, tick=%dms\n\n", ph, tick_ms);

  // Inicializa o randomizador
  srand(time(NULL));

  // Inicializa o Clock
  if (clock_gettime(CLOCK_REALTIME, &ultimo_update) < 0) {
    printf("Erro em clock_gettime");
    return -1;
  }

  // Inicializa as primitivas de sincronização
  pthread_mutex_init(&mutex, NULL);
  pthread_cond_init(&H_cond, NULL);
  pthread_cond_init(&O_cond, NULL);
  pthread_barrier_init(&barrier, NULL, 3);
  pthread_mutex_init(&barrier_mutex, NULL);

  while (1) {
    // Loop Principal

    if (clock_gettime(CLOCK_REALTIME, &esse_update) < 0) {
      printf("Erro em clock_gettime");
      return -1;
    }

    // Checar Update e sair se não atingiu o parametro tick
    long tempo_decorrido_ms = (esse_update.tv_sec - ultimo_update.tv_sec) * 1000 +
                              (esse_update.tv_nsec - ultimo_update.tv_nsec) / 1000000;
    if (tick_ms != 0 && tempo_decorrido_ms < tick_ms) {
      usleep(1000);
      continue;
    }

    // A partir desse ponto, começa a geração de um átomo
    ultimo_update = esse_update;

    // Gera um número aleatório para decidir H ou O
    int r = rand() % 100;

    // Probabilidade baseada em ph
    // Se ph=2, queremos 2 H para cada 1 O, ou seja, 66% H, 33% O
    int prob_h = (ph * 100) / (ph + 1);

    if (r < prob_h) {
      // Cria thread Hidrogênio
      int* id = malloc(sizeof(int));
      *id = H_id++;
      if (pthread_create(&thread, NULL, hydrogen, id) != 0) {
        perror("Erro ao criar thread hydrogen");
        free(id);
        } else {
          pthread_detach(thread); // Detach para não precisar fazer join
        }
    } else {
      // Cria thread Oxigênio
      int* id = malloc(sizeof(int));
      *id = O_id++;
      if (pthread_create(&thread, NULL, oxygen, id) != 0) {
        perror("Erro ao criar thread oxygen");
        free(id);
      } else {
        pthread_detach(thread); // Detach para não precisar fazer join
      }
    }
  }

  return 0;
}


In [ ]:
!gcc -o h2o_final h2o_final.c -lpthread

In [ ]:
!./h2o_final 2 50

# Building H2O: Versão Final

## Espera Atômica

O maior obstáculo desse desafio de sincronização foi o desengate do mutex seguido pela espera em semáforo em uma sequência não atômica. Isso pode levar a criação de moléculas mal formadas (Apenas 1 O e 1 H) e vazamento de memória devido ao contador e o tamanho da fila de espera estarem desincronizados.

```c
// Não pode formar ainda, precisa esperar
pthread_mutex_unlock(&mutex);

// Espera ser sinalizado
sem_wait(&H_sem);
```
A solução foi transformar estas linhas em uma operação atômica, o qual foi possível usando outra primitiva. Ao invés de `sem_t`, usamos `pthread_cond_t`, que nos permite fazer com que a espera de uma thread seja atrelado com um mutex. Esta função atomicamente libera o mutex e coloca a thread em espera,
eliminando a condição de corrida.

Note como o código não mudou muito de sua versão inicial.
```c
void* hydrogen(void* arg) {
    int id = *(int*)arg;

    pthread_mutex_lock(&mutex);
    H_count++;
    printf("H%d chega. Esperando: H=%d, O=%d\n", id, H_count, O_count);

    if (H_count >= 2 && O_count >= 1) {
        // Inicia a formação dá molécula
        H_count -= 2;
        O_count -= 1;
        printf("H%d: Formando molécula! Novos valores: H=%d, O=%d\n", id, H_count, O_count);

        // Acorda as outras threads
        pthread_cond_signal(&H_cond);
        pthread_cond_signal(&O_cond);

        pthread_mutex_unlock(&mutex);
    } else {
        // Não há threads o suficiente, espera
        pthread_cond_wait(&H_cond, &mutex);
        // Ao acordar, desliga o mutex se não trava ao tentar formar molécula
        pthread_mutex_unlock(&mutex);
    }

    barreira(HYDROGEN);

    free(arg);
    return NULL;
}
```

## Lógica de Barreira

Entretanto, há outra mudança para maior segurança de sincronização. A função `barreira()`. Esta é chamada usando como parametro uma das constantes HYDROGEN ou OXYGEN, apenas para transmitir qual tipo de thread está chamando a função.

Dentro dela, temos mais um mutex e mais dois contadores. Estes contadores fazem parte da lógica da barreira em assegurar que estamos certeiramente formando uma molécula H2O, ocorrendo uma falha se estiverem errados.
```c
void barreira(int type) {
    pthread_mutex_lock(&barrier_mutex);
    if (type == OXYGEN)
        barrier_O_count++;
    else
        barrier_H_count++;
    pthread_mutex_unlock(&barrier_mutex);

    int result = pthread_barrier_wait(&barrier);

    if (result == PTHREAD_BARRIER_SERIAL_THREAD) {
        pthread_mutex_lock(&barrier_mutex);

        if (barrier_H_count != 2 || barrier_O_count != 1) {
            printf("ERRO FATAL! Composição incorreta: H=%d, O=%d\n",
                   barrier_H_count, barrier_O_count);
            pthread_mutex_unlock(&barrier_mutex);
            exit(1);
        }

        barrier_H_count = 0;
        barrier_O_count = 0;

        pthread_mutex_unlock(&barrier_mutex);

        bond();
    }
}
```
O importante é a seção abaixo. Nesse bloco `if`, somente entra **uma única thread**, a última a entrar na barreira, garantido pelo retorno `PTHREAD_BARRIER_SERIAL_THREAD` de `pthread_barrier_wait(&barrier)`.
```c
if (result == PTHREAD_BARRIER_SERIAL_THREAD) {
    pthread_mutex_lock(&barrier_mutex);

    if (barrier_H_count != 2 || barrier_O_count != 1) {
        printf("ERRO FATAL! Composição incorreta: H=%d, O=%d\n",
                   barrier_H_count, barrier_O_count);
        pthread_mutex_unlock(&barrier_mutex);
        exit(1);
    }

    barrier_H_count = 0;
    barrier_O_count = 0;

    pthread_mutex_unlock(&barrier_mutex);

    bond();
}
```

# Apresentação em Slides

https://docs.google.com/presentation/d/1tPoYEUW3MFopXHi4bD_iR7_LZdb_BPScomjl3KYgDos/edit?usp=sharing

A apresentação antiga termina no slide 7. Qualquer slide a partir do 8 são novos.